# From Waveform to Genre — Analysis & Visualization


**FMA dataset (citation)**  
Defferrard, M., Benzi, K., Vandergheynst, P., & Bresson, X. (2017). *FMA: A Dataset for Music Analysis*. 18th ISMIR. PDF: https://arxiv.org/pdf/1612.01840.pdf

**License / data note:** FMA metadata is CC BY 4.0; audio files follow per-artist Creative Commons licenses. Consult the original dataset for terms of use.

---

### Project adaptation note
This notebook adapts the FMA data pipeline for the current workflow and interactive environments (Google Colab / Drive). Key adaptations include Colab/Drive path variables, idempotent download & extraction steps, automatic manifest generation for reproducibility, and helper utilities for checksums and seeding. Any reuse of original code or logic from the mdeff/fma project is indicated in the header and documented in the repository.

### Short comparison vs. original mdeff/fma
This notebook focuses on model inference, evaluation and visualization for the trained genre classifier. Compared to the original mdeff/fma repository, this notebook:

- Automates inference on the test set and saves `inference_results.csv` (predictions and correctness flags).
- Extracts and stores latent embeddings (`latent_representations.npy`) for downstream analysis and visualization.
- Generates annotated evaluation artifacts: confusion matrix PNG, classification_report JSON, learning curves, and t-SNE/UMAP plots.
- Includes an error-analysis workflow that saves spectrogram images for misclassified examples to aid debugging.
- Adds Colab/Drive integration and artifact synchronization for reproducibility (search for latest model, save weights and reports to Drive).

**Conclusion:** The notebook reuses core ideas from mdeff/fma (audio handling and baseline evaluation) but extends them with automation, richer diagnostics, and Colab-oriented tooling.

***
**Purpose:** This notebook is designed to evaluate the trained model and analyze its decision-making process through advanced visualization. It identifies the latest model in `LOCAL_DATA` and performs a complete inference pass on the test set to generate `inference_results.csv`.

The analysis further includes extracting feature embeddings to save as `latent_representations.npy`, alongside computing a detailed confusion matrix and t-SNE plot to visualize genre separation. Finally, the workflow generates a `feature_analysis_meta.json` manifest and synchronizes all generated results back to **Google Drive** for long-term project persistence.

### Workspace Configuration and Environment Setup

The script mounts Google Drive (if running in Colab), defines the primary project directories, creates the necessary local folders, and subsequently switches the current working directory to them.

In [ ]:
# mount Drive (Colab). If not running in Colab, mount will be skipped gracefully.
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except Exception:
    print("google.colab.drive.mount skipped (not running in Colab or import failed).")

import os, shutil

WORKDIR = "/content/waveform_genre_project"
LOCAL_DATA = os.path.join(WORKDIR, "data_storage")
DRIVE_ROOT = "/content/drive/MyDrive/waveform_analysis_outputs"

# ensure WORKDIR and LOCAL_DATA exist
os.makedirs(LOCAL_DATA, exist_ok=True)

# change working directory
os.chdir(WORKDIR)

print("WORKDIR:", WORKDIR)
print("LOCAL_DATA:", LOCAL_DATA)
print("DRIVE_ROOT:", DRIVE_ROOT)
print("WORKDIR contents (first 20):", os.listdir(WORKDIR)[:20])

This code synchronizes data from Google Drive to the local Colab environment by calculating the total file count, copying only missing items, and displaying real-time progress while tracking successful transfers, skips, and errors.

In [ ]:
import os, shutil, sys

# Define Source (Drive) and Destination (Local Colab) paths
SRC = "/content/drive/MyDrive/waveform_analysis_outputs/data_storage"
DST = "/content/waveform_genre_project/data_storage"

# Create destination directory if it doesn't exist
os.makedirs(DST, exist_ok=True)

# 1) Calculate total number of files for progress tracking
total_files = 0
for _, _, files in os.walk(SRC):
    total_files += len(files)

if total_files == 0:
    print("В SRC няма файлове за копиране:", SRC)
else:
    copied = 0
    skipped = 0
    errors = []

    processed = 0
    # 2) Traverse and copy files
    for root, dirs, files in os.walk(SRC):
        rel = os.path.relpath(root, SRC)
        target_dir = os.path.join(DST, rel) if rel != "." else DST
        os.makedirs(target_dir, exist_ok=True)
        for f in files:
            processed += 1
            src_file = os.path.join(root, f)
            dst_file = os.path.join(target_dir, f)
            try:
                if not os.path.exists(dst_file):
                    # Only copy if the file does not already exist locally (Incremental sync)
                    shutil.copy2(src_file, dst_file)
                    copied += 1
                else:
                    skipped += 1
            except Exception as e:
                errors.append((src_file, str(e)))

            # 3) Update progress every 20 files or at the end
            if processed % 20 == 0 or processed == total_files:
                pct = int(processed / total_files * 100)
                # Use carriage return (\r) to overwrite the line in the console
                sys.stdout.write(f"\rProcessed: {processed}/{total_files} ({pct}%)  Copied: {copied}  Skipped: {skipped}  Errors: {len(errors)}")
                sys.stdout.flush()


    print()
    # Final summary print
    print("Copying is complete.")
    print(f"Total files: {total_files}, copied new: {copied}, skipped (already available): {skipped}, errors: {len(errors)}")
    # Report first 5 errors if any occurred
    if errors:
        print("The first 5 mistakes (src, error):")
        for e in errors[:5]:
            print(" -", e[0], ":", e[1])

    # Show multiple files in the local target directory
    try:
        sample = os.listdir(DST)[:50]
        print("First files in local folder:", sample)
    except Exception:
        pass